## Sentence Transformer

In [1]:
# 1. Load pretrained model
from sentence_transformers import SentenceTransformer
import json

model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/rayan/learning/git/agentic-rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17097.93it/s]


In [8]:
# load json dataset
with open('recipes_dataset.json', 'r') as f:
    data = json.load(f)

print(len(data))

print(data[0])

55
{'id': 1, 'name': 'Classic Margherita Pizza', 'cuisine': 'Italian', 'ingredients': ['2 cups all-purpose flour', '1 teaspoon instant yeast', '1 teaspoon salt', '3/4 cup warm water', '1 tablespoon olive oil', '1/2 cup tomato sauce', '1 cup fresh mozzarella cheese', 'Fresh basil leaves', 'Olive oil for drizzling'], 'instructions': 'Mix flour, yeast, salt, and water to form a dough. Knead for 10 minutes, then let rise for 1 hour. Roll out dough and spread tomato sauce. Add mozzarella cheese and fresh basil. Bake at 475°F (245°C) for 12-15 minutes. Drizzle with olive oil before serving.', 'cooktime_minutes': 65, 'nutrition': {'calories': 320, 'protein': 12, 'carbohydrates': 45, 'fat': 14, 'fiber': 3, 'sodium': 620}}


In [ ]:
for recipe in data:
    recipe['embedding'] = f"recipe name- {recipe['name']}; cooktime in minutes- {recipe['cooktime_minutes']}; cuisine- {recipe['cuisine']}; nutrition info- {', '.join(f'{k}: {v}' for k, v in recipe['nutrition'].items())}; ingredients- {', '.join(recipe['ingredients'])}; instructions- {recipe['instructions']}"
    print (recipe['embedding'])

In [ ]:
def get_metadata(recipe):
    return {
        # "id": recipe['id'],
        "name": recipe['name'],
        "cooktime_minutes": recipe['cooktime_minutes'],
        "cuisine": recipe['cuisine'],
        "nutrition": json.dumps(recipe['nutrition']),
        "ingredients": ", ".join(recipe['ingredients']),
        # "instructions": recipe['instructions']
    }

embeddings = []
ids=[]
documents = []
metadata = []

for recipe in data:
    ids.append(f"id_{recipe['id']}")
    text_to_encode = f"recipe name- {recipe['name']}; cooktime in minutes- {recipe['cooktime_minutes']}; cuisine- {recipe['cuisine']}; nutrition info- {', '.join(f'{k}: {v}' for k, v in recipe['nutrition'].items())}; ingredients- {', '.join(recipe['ingredients'])}; instructions- {recipe['instructions']}"
    documents.append(text_to_encode)
    embedding = model.encode(text_to_encode).tolist()
    embeddings.append(embedding)
    metadata.append(get_metadata(recipe))



In [40]:
# print(json.dumps(embeddings[0], indent=2))
print(len(embeddings[0]))
print(len(json.dumps(embeddings[0], indent=2)))
print(json.dumps(embeddings[0], indent=2)[:500])

384
9188
[
  -0.10508333146572113,
  0.028948985040187836,
  -0.05876011401414871,
  0.05328354984521866,
  -0.060586828738451004,
  0.022325359284877777,
  -0.06919200718402863,
  0.02743317373096943,
  -0.01038674172013998,
  -0.16737790405750275,
  -0.01954864338040352,
  -0.021840998902916908,
  0.022136017680168152,
  -0.08418310433626175,
  -0.03794889897108078,
  -0.08516929298639297,
  0.04048050567507744,
  0.02853826805949211,
  -0.03469318151473999,
  -0.03968004509806633,
  0.0493665337562561


In [51]:
# initialize chroma client
import chromadb

client = chromadb.Client()
if 'recipe-embeddings' in [collection.name for collection in client.list_collections()]:
    collection = client.delete_collection('recipe-embeddings')
    print(f"Deleted existing collection 'recipe-embeddings'.")

collection = client.create_collection('recipe-embeddings')
print(f"Created new collection 'recipe-embeddings'.")

print ([collection.name for collection in client.list_collections()])
# add embeddings to collection
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadata
)

print(f"Added {len(embeddings)} embeddings with average {len(embeddings[0])} dimensions for {len(documents)} documents to collection 'recipe-embeddings'.")

Deleted existing collection 'recipe-embeddings'.
Created new collection 'recipe-embeddings'.
['recipe-embeddings']
Added 55 embeddings with average 384 dimensions for 55 documents to collection 'recipe-embeddings'.


In [63]:
user_input = "I want a Indian cuisine made with rice."
# encode user input
user_embedding = model.encode(user_input).tolist()

retrieval_results = collection.query(
    query_embeddings=[user_embedding],
    n_results=2,
    # where={"cuisine": "Indian"})
    where={
        "$and": [
            {"cuisine": {"$eq": "Indian"}},
            {"ingredients": {"$contains": "rice"}}
        ]
    }
)
retrieved_documents = retrieval_results['documents'][0]
print("Retrieved documents:")
for i, doc in enumerate(retrieved_documents):
    print(f"{i+1}. {doc}")



Retrieved documents:


In [64]:
# Visualize the recipe data in the vector database
import pandas as pd
import numpy as np

# Get all data from the collection
results = collection.get(include=['documents', 'metadatas', 'embeddings'])

print(f"Total recipes in collection: {len(results['ids'])}")
print()

# Create a DataFrame for easier viewing
embeddings = results['embeddings']
if embeddings is not None and len(embeddings) > 0:
    embedding_dims = [len(e) for e in embeddings]
else:
    embedding_dims = [0] * len(results['ids'])

df = pd.DataFrame({
    'ID': results['ids'],
    'Document': results['documents'],
    'Embedding_Dim': embedding_dims
})

# Display first few rows
print("First 5 recipes:")
print(df[['ID', 'Embedding_Dim']].head())
print()

# Show a sample document (truncated for readability)
print("Sample document (first 500 chars):")
print(results['documents'][0][:500])
print("...")

# Count recipes by cuisine (extract from document text)
cuisines = []
# for doc in results['documents']:
#     # Extract cuisine from the document string
#     if 'cuisine-' in doc:
#         cuisine = doc.split('cuisine-')[1].split(';')[0].strip()
#         cuisines.append(cuisine)

for cuisine in [recipe['cuisine'] for recipe in data]:
    cuisines.append(cuisine)

cuisine_counts = pd.Series(cuisines).value_counts()
print("\nRecipes by cuisine:")
print(cuisine_counts)

# Show embedding statistics
if embeddings is not None and len(embeddings) > 0:
    embeddings_array = np.array(embeddings)
    print(f"\nEmbedding statistics:")
    print(f"  Shape: {embeddings_array.shape}")
    print(f"  Mean: {embeddings_array.mean():.4f}")
    print(f"  Std: {embeddings_array.std():.4f}")
    print(f"  Min: {embeddings_array.min():.4f}")
    print(f"  Max: {embeddings_array.max():.4f}")

Total recipes in collection: 55

First 5 recipes:
     ID  Embedding_Dim
0  id_1            384
1  id_2            384
2  id_3            384
3  id_4            384
4  id_5            384

Sample document (first 500 chars):
recipe name- Classic Margherita Pizza; cooktime in minutes- 65; cuisine- Italian; nutrition info- calories: 320, protein: 12, carbohydrates: 45, fat: 14, fiber: 3, sodium: 620; ingredients- 2 cups all-purpose flour, 1 teaspoon instant yeast, 1 teaspoon salt, 3/4 cup warm water, 1 tablespoon olive oil, 1/2 cup tomato sauce, 1 cup fresh mozzarella cheese, Fresh basil leaves, Olive oil for drizzling; instructions- Mix flour, yeast, salt, and water to form a dough. Knead for 10 minutes, then let ris
...

Recipes by cuisine:
Italian              6
Indian               5
American             5
Japanese             5
Mexican              5
Middle Eastern       5
Thai                 4
French               4
Chinese              2
Asian                2
Greek               

## Data Visualization

In [ ]:
cuisines = set(recipe['cuisine'] for recipe in data)
print("Available cuisines:")
for cuisine in cuisines:
    print(f"- {cuisine}")

In [65]:
# 3D Visualization of Chroma Vector Store using t-SNE
import plotly.graph_objects as go
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import numpy as np

# Get embeddings from collection
results = collection.get(include=['documents', 'embeddings'])
embeddings = np.array(results['embeddings'])
documents = results['documents']
ids = results['ids']

# Extract cuisine labels for coloring
cuisines = []
for doc in documents:
    if 'cuisine-' in doc:
        cuisine = doc.split('cuisine-')[1].split(';')[0].strip()
        cuisines.append(cuisine)
    else:
        cuisines.append('Unknown')

# Reduce to 3D using t-SNE (better for visualization) or PCA (faster)
print("Reducing dimensions with t-SNE...")
tsne = TSNE(n_components=3, random_state=42, perplexity=min(30, len(embeddings)-1))
embeddings_3d = tsne.fit_transform(embeddings)

# Alternative: PCA (uncomment to use instead)
# print("Reducing dimensions with PCA...")
# pca = PCA(n_components=3)
# embeddings_3d = pca.fit_transform(embeddings)

# Create color mapping for cuisines
unique_cuisines = list(set(cuisines))
color_map = {cuisine: i for i, cuisine in enumerate(unique_cuisines)}
colors = [color_map[c] for c in cuisines]

# Create hover text with recipe info
hover_texts = []
for i, (doc, id_) in enumerate(zip(documents, ids)):
    # Extract key info for hover
    name = doc.split('recipe name-')[1].split(';')[0].strip() if 'recipe name-' in doc else 'Unknown'
    cuisine = cuisines[i]
    cooktime = doc.split('cooktime in minutes-')[1].split(';')[0].strip() if 'cooktime in minutes-' in doc else 'Unknown'
    hover_texts.append(f"ID: {id_}<br>Name: {name}<br>Cuisine: {cuisine}<br>Cook time: {cooktime} min")

# Create 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=embeddings_3d[:, 0],
    y=embeddings_3d[:, 1],
    z=embeddings_3d[:, 2],
    mode='markers',
    marker=dict(
        size=6,
        color=colors,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Cuisine Index"),
        opacity=0.8
    ),
    text=hover_texts,
    hoverinfo='text',
    name='Recipes'
)])

fig.update_layout(
    title='3D Visualization of Recipe Embeddings (t-SNE)',
    scene=dict(
        xaxis_title='t-SNE 1',
        yaxis_title='t-SNE 2',
        zaxis_title='t-SNE 3'
    ),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

# Also create a 2D version for comparison
print("\nCreating 2D visualization...")
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
embeddings_2d = tsne_2d.fit_transform(embeddings)

fig2d = go.Figure(data=[go.Scatter(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    mode='markers',
    marker=dict(
        size=8,
        color=colors,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Cuisine Index"),
        opacity=0.8
    ),
    text=hover_texts,
    hoverinfo='text'
)])

fig2d.update_layout(
    title='2D Visualization of Recipe Embeddings (t-SNE)',
    xaxis_title='t-SNE 1',
    yaxis_title='t-SNE 2',
    width=800,
    height=600
)

fig2d.show()

Reducing dimensions with t-SNE...



Creating 2D visualization...
